In [1]:
import sys
import pandas as pd
from pathlib import Path
pd.set_option('display.max_columns', None)
sys.path.append(str(Path.cwd().parent / "src"))
import csv
import numpy
import utils.data_loader as dl
import utils.data_summaries as ds
import utils.standardize_columns as sc

In [2]:
#print(dir(dl))
#print(dir(ds))
print(dir(sc))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_resolve', 'col_to_bool', 'col_to_cat', 'col_to_date', 'col_to_float', 'col_to_int', 'col_to_snake', 'col_to_str', 'pd', 're']


In [3]:
#Read available datasets available for data source
path = "C:\\Users\\1\\Desktop\\project\\data\\cleaned_data"
DataSources = dl.get_sources(path)
print(DataSources)

[WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/311'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/climate'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/neighbourhoods'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/watermain_breaks'), WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/watermains')]


In [4]:
source = DataSources[0]
readable = dl.read_files(source)
readable

CSV
  311_requests_2010_2012.csv
  311_requests_2013_2015.csv
  311_requests_2016_2018.csv
  311_requests_clean.csv


[WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/311/311_requests_2010_2012.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/311/311_requests_2013_2015.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/311/311_requests_2016_2018.csv'),
 WindowsPath('C:/Users/1/Desktop/project/data/cleaned_data/311/311_requests_clean.csv')]

In [35]:
data_311 = dl.pull_data(source)
print(f"number of datasets from source: {len(data_311)}")
print(f"names of remaining datasets: {data_311.keys()}")

number of datasets from source: 4
names of remaining datasets: dict_keys(['311_requests_2010_2012', '311_requests_2013_2015', '311_requests_2016_2018', '311_requests_clean'])


In [36]:
data_311['311_requests_clean']

,creation_date,status,first_3_chars_of_postal_code,intersection_street_1,intersection_street_2,ward,service_request_type,division,section,ward_25
0,2010-01-01 00:38:26,Closed,Intersection,High Park Blvd,Parkside Dr,Parkdale-High Park (13),Road - Sanding / Salting Required,Transportation Services,Road Operations,4.0
1,2010-01-01 01:19:18,Closed,M4T,NaN,NaN,Toronto Centre-Rosedale (27),Water Service Line-Turn On,Toronto Water,District Ops,11.0
2,2010-01-01 01:19:48,Cancelled,M1T,NaN,NaN,Scarborough-Agincourt (40),Water Service Line-Leaking,Toronto Water,District Ops,22.0
3,2010-01-01 01:24:18,Cancelled,M1T,NaN,NaN,Scarborough-Agincourt (40),Water Service Line-Leaking,Toronto Water,District Ops,22.0
4,2010-01-01 01:42:26,Closed,M1K,NaN,NaN,Scarborough Southwest (35),Watermain-Possible Break,Toronto Water,District Ops,20.0
...,...,...,...,...,...,...,...,...,...,...
3341854,2018-12-31 23:30:12,In-progress,M6R,NaN,NaN,Parkdale-High Park (04),Noise,Municipal Licensing & Standards,District Enforcement,4.0
3341855,2018-12-31 23:31:17,Closed,M1R,NaN,NaN,Scarborough Centre (21),Noise,Municipal Licensing & Standards,District Enforcement,21.0
3341856,2018-12-31 23:31:48,Closed,M6K,NaN,NaN,Parkdale-High Park (04),Noise,Municipal Licensing & Standards,District Enforcement,4.0
3341857,2018-12-31 23:52:31,Closed,M6P,NaN,NaN,Parkdale-High Park (04),ENF/INVEST ANIM CARE,Municipal Licensing & Standards,Toronto Animal Services,4.0


In [37]:
#ds.full_summary(data_311)
#ds.quick_summary(data_311['311_requests_clean'])

In [49]:
#Majority of missing fields are intersection, we don't need that we already have the ward, drop them, probably won't need postal code either
requests311 = data_311['311_requests_clean'].drop(columns=["first_3_chars_of_postal_code","intersection_street_1","intersection_street_2"])

In [39]:
#requests311 = data_311['311_requests_clean']
ds.quick_summary(requests311) #no more missing fields

,dtype,missing_pct,n_unique
creation_date,object,0.000000e+00,3301214
status,object,0.000000e+00,6
first_3_chars_of_postal_code,object,0.000000e+00,101
ward,object,0.000000e+00,69
service_request_type,object,0.000000e+00,619
division,object,0.000000e+00,7
section,object,0.000000e+00,20
ward_25,float64,2.992346e-07,25


In [51]:
requests311["service_request_type"].value_counts()

service_request_type
Res / Organic Bin / Replace Damaged      117428
Property Standards                       113351
Residential Furniture / Not Picked Up     99456
Sewer Service Line-Blocked                87375
Road - Pot hole                           84602
                                          ...  
TAS_COYOT RESP HU-BITE                        1
OWNER SURR DECEASED                           1
SERVICES COMM EVENT                           1
Angle Parking                                 1
Development Applications                      1
Name: count, Length: 619, dtype: int64

In [52]:
flood_h = [
    # --- HIGH ---
    "Catch Basin - Blocked / Flooding",
    "Driveway - Damaged / Ponding",
    "Road Water Ponding",
    "Sidewalk Water Ponding",
    "Watercourses-Blocked/Flooding",
    "Watercourses-Erosion/Washout",
    "Watercourses-Outfalls/Inlets",
    "West Nile Virus - Standing water / Roadway",
    "West Nile Virus-Standing water / Roadside",
    "Storm Event-Flooding"
]
flood_m = [
    # --- MEDIUM ---
    "Watermain-Possible Break",
    "Watermain Valve - Turn On",
    "Salting-Winter (WSL/HYDT/VALVE/Watermain Break Locations etc.)",
    "Hydrant-Leaking",
    "Water Valve-Leaking",
    "Water Service Line-Damaged Water Service Box"
]
patternh = "|".join(flood_h)
patternm = "|".join(flood_m)
pattern_all = "|".join(flood_h + flood_m)
flood = ds.search_df(requests311,pattern_all,col_name = 'service_request_type')
flood

C:\Users\1\Desktop\project\src\utils\data_summaries.py:81: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = col.str.contains(term, case=False, na=False, regex=True)


,creation_date,status,ward,service_request_type,division,section,ward_25
4,2010-01-01 01:42:26,Closed,Scarborough Southwest (35),Watermain-Possible Break,Toronto Water,District Ops,20.0
22,2010-01-01 09:49:09,Closed,Toronto Centre-Rosedale (27),Watermain-Possible Break,Toronto Water,District Ops,11.0
24,2010-01-01 09:58:58,Cancelled,Toronto Centre-Rosedale (27),Watermain-Possible Break,Toronto Water,District Ops,11.0
25,2010-01-01 10:03:29,Cancelled,Toronto Centre-Rosedale (28),Watermain-Possible Break,Toronto Water,District Ops,13.0
48,2010-01-01 13:33:31,Closed,Etobicoke Centre (04),Watermain-Possible Break,Toronto Water,District Ops,2.0
...,...,...,...,...,...,...,...
3341683,2018-12-31 14:37:19,Closed,University-Rosedale (11),Watermain-Possible Break,Toronto Water,District Ops,11.0
3341785,2018-12-31 17:23:40,Closed,Scarborough-Agincourt (22),Catch Basin - Blocked / Flooding,Transportation Services,Road Operations,22.0
3341798,2018-12-31 18:53:30,Closed,Etobicoke-Lakeshore (03),Catch Basin - Blocked / Flooding,Transportation Services,Road Operations,3.0
3341840,2018-12-31 22:31:03,Closed,Don Valley East (16),Road Water Ponding,Transportation Services,Road Operations,16.0


In [70]:
ds.check_duplicates(flood,["creation_date","status","ward","service_request_type"]) #check duplicates to determine if safe to remove. They are safe, we will remove them
flood_dupeless = flood.drop_duplicates(subset=["creation_date","status","ward","service_request_type"], keep="first").reset_index(drop=True)
#remove the duplicates, keeping the first.
#flood_dupeless = flood.drop_duplicates(subset=["creation_date","first_3_chars_of_postal_code", "ward","service_request_type"], keep="first").reset_index(drop=True)
ds.check_duplicates(flood_dupeless,["creation_date","ward","service_request_type"]) #check duplicates to determine if safe to remove. Remaining are cancelled, we can drop these
flood_dupeless = flood_dupeless.drop_duplicates(subset=["creation_date","ward","service_request_type"], keep=False).reset_index(drop=True)

In [73]:
#requests are said to be cancelled if request was not worked, remove cancelled requests
flood_dupeless = flood_dupeless[flood_dupeless["status"] != "Cancelled"]
ds.quick_summary(flood_dupeless)

,dtype,missing_pct,n_unique
creation_date,object,0.0,77796
status,object,0.0,3
ward,object,0.0,68
service_request_type,object,0.0,14
division,object,0.0,2
section,object,0.0,2
ward_25,float64,0.0,25


In [74]:
#combine ward with ward_25 into one column
import re
flood_ward = flood_dupeless
flood_ward["ward"] = flood_dupeless.apply(
    lambda x: re.sub(r"\(\d+\)", f"({int(x['ward_25'])})", x["ward"]), axis=1
)
flood_ward = flood_ward.drop(columns='ward_25')
flood_ward

,creation_date,status,ward,service_request_type,division,section
0,2010-01-01 01:42:26,Closed,Scarborough Southwest (20),Watermain-Possible Break,Toronto Water,District Ops
1,2010-01-01 09:49:09,Closed,Toronto Centre-Rosedale (11),Watermain-Possible Break,Toronto Water,District Ops
4,2010-01-01 13:33:31,Closed,Etobicoke Centre (2),Watermain-Possible Break,Toronto Water,District Ops
5,2010-01-01 15:23:57,Closed,Willowdale (18),Watermain-Possible Break,Toronto Water,District Ops
6,2010-01-01 18:29:17,Closed,York South-Weston (5),Catch Basin - Blocked / Flooding,Transportation Services,Road Operations
...,...,...,...,...,...,...
82885,2018-12-31 14:37:19,Closed,University-Rosedale (11),Watermain-Possible Break,Toronto Water,District Ops
82886,2018-12-31 17:23:40,Closed,Scarborough-Agincourt (22),Catch Basin - Blocked / Flooding,Transportation Services,Road Operations
82887,2018-12-31 18:53:30,Closed,Etobicoke-Lakeshore (3),Catch Basin - Blocked / Flooding,Transportation Services,Road Operations
82888,2018-12-31 22:31:03,Closed,Don Valley East (16),Road Water Ponding,Transportation Services,Road Operations


In [78]:
print(flood_ward.dtypes)
##begin casting dtypes:
print(dir(sc))
sc.col_to_cat(flood_ward, ["status", "ward","service_request_type", "division", "section"])
sc.col_to_date(flood_ward,["creation_date"])

flood_ward

creation_date           datetime64[ns]
status                        category
ward                          category
service_request_type          category
division                      category
section                       category
dtype: object
['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_resolve', 'col_to_bool', 'col_to_cat', 'col_to_date', 'col_to_float', 'col_to_int', 'col_to_snake', 'col_to_str', 'pd', 're']


,creation_date,status,ward,service_request_type,division,section
0,2010-01-01,Closed,Scarborough Southwest (20),Watermain-Possible Break,Toronto Water,District Ops
1,2010-01-01,Closed,Toronto Centre-Rosedale (11),Watermain-Possible Break,Toronto Water,District Ops
4,2010-01-01,Closed,Etobicoke Centre (2),Watermain-Possible Break,Toronto Water,District Ops
5,2010-01-01,Closed,Willowdale (18),Watermain-Possible Break,Toronto Water,District Ops
6,2010-01-01,Closed,York South-Weston (5),Catch Basin - Blocked / Flooding,Transportation Services,Road Operations
...,...,...,...,...,...,...
82885,2018-12-31,Closed,University-Rosedale (11),Watermain-Possible Break,Toronto Water,District Ops
82886,2018-12-31,Closed,Scarborough-Agincourt (22),Catch Basin - Blocked / Flooding,Transportation Services,Road Operations
82887,2018-12-31,Closed,Etobicoke-Lakeshore (3),Catch Basin - Blocked / Flooding,Transportation Services,Road Operations
82888,2018-12-31,Closed,Don Valley East (16),Road Water Ponding,Transportation Services,Road Operations


In [81]:
ds.quick_summary(flood_ward)

,dtype,missing_pct,n_unique
creation_date,datetime64[ns],0.0,3285
status,category,0.0,3
ward,category,0.0,32
service_request_type,category,0.0,14
division,category,0.0,2
section,category,0.0,2


In [79]:
flood_ward["creation_date"].nunique() / len(flood_dupeless) #checks whether a column should be category, if score < 0.01 it is acceptable

0.04219924208362772

In [80]:
flood_ward["first_3_chars_of_postal_code"].value_counts()

KeyError: 'first_3_chars_of_postal_code'